# Preparación de Datos — Siniestros Viales, Municipio de Envigado

**Fuente:** Datos Abiertos Colombia — Accidentalidad Municipio de Envigado
**Variable objetivo:** `GRAVEDAD` — se modela como problema de **clasificación binaria** (SOLO DAÑOS / HERIDOS); la clase MUERTOS se descarta en el paso 8
**Registros:** > 43.000 siniestros únicos · **Variables predictoras finales:** 8

Fases desarrolladas:
1. Integración de los datos
2. Eliminar variables irrelevantes/redundantes
3. Descripción estadística de los datos
4. Limpieza de datos atípicos
5. Limpieza de datos nulos
6. Análisis de correlaciones para redundancias
7. Análisis de correlaciones para irrelevancias
8. Balanceo de datos
9. Transformaciones (discretización / normalización)
10. Guardar los datos preparados
11. Modelos de clasificación con hiperparametrización (GridSearchCV + validación cruzada)

In [ ]:
# Importamos librerías básicas
import pandas as pd            # manipulación de dataframes
import numpy as np             # matrices y vectores
import matplotlib.pyplot as plt  # gráficas
import seaborn as sns          # mapas de calor / gráficas estadísticas
%matplotlib inline

pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (8, 5)

# 1. Integración de los datos

El dataset proviene de una única fuente (portal de datos abiertos), pero **no está integrado de forma homogénea**:
- El mismo siniestro (`RADICADO`) aparece repetido varias veces (una fila por cada vehículo/actor involucrado), por lo que no hay una fila = un accidente.
- Las columnas `FECHA` y `HORA` combinan **dos formatos distintos** dentro de la misma columna (probablemente por venir de dos exportaciones/sistemas distintos que se concatenaron en un mismo archivo). Esto es exactamente un problema de *integración de datos*: hay que reconciliar los formatos antes de poder usarlos.

Resolvemos ambos problemas en esta fase.

In [ ]:
# Cargamos los datos
data = pd.read_csv("Accidentalidad_Municipio_de__Envigado_20260929.csv", low_memory=False)
print("Filas x Columnas:", data.shape)
data.head()

In [ ]:
data.info()

## 1.1 Un accidente = varias filas (una por vehículo/persona involucrada)

Revisamos cuántas veces se repite cada `RADICADO` (identificador del siniestro).

In [ ]:
# Revisamos si tenemos RADICADOS repetidos
ids = data['RADICADO'].value_counts()
print("RADICADOS únicos:", data['RADICADO'].nunique(), "de", len(data), "filas")
ids[ids > 1].head()

In [ ]:
# Consolidamos: un siniestro = un registro. Nos quedamos con la versión más reciente de cada RADICADO
data.drop_duplicates(subset=['RADICADO'], keep='last', inplace=True)
print("Shape tras consolidar por RADICADO:", data.shape)

In [ ]:
# Verificamos que ya no hay RADICADOS repetidos
ids = data['RADICADO'].value_counts()
ids[ids > 1]

## 1.2 `FECHA` y `HORA`: formatos mixtos

Al inspeccionar `FECHA` encontramos registros como `'1/01/2014'` (texto `DD/MM/AAAA`) mezclados con `'2022-06-22T00:00:00.000'` (formato ISO). Pandas puede resolver esto con `format='mixed'`.

`HORA` es más problemático: mezcla un formato de texto en español ("10:30:00 p. m.") con un formato tipo fecha-Excel ("1899-12-31T14:30:00.000", donde 1899-12-31 es la fecha "cero" de Excel al exportar solo la hora). Escribimos una función que reconoce cada patrón y devuelve la hora correcta en ambos casos.

In [ ]:
# Ejemplo de los dos formatos que conviven en FECHA
print("Formato 1:", data['FECHA'].iloc[0])
print("Formato 2:", data.loc[data['FECHA'].str.startswith('2022'), 'FECHA'].iloc[0])

# Ejemplo de los dos formatos que conviven en HORA
print("\nFormato 1 (texto AM/PM):", data.loc[~data['HORA'].str.startswith('1899'), 'HORA'].iloc[0])
print("Formato 2 (pseudo-fecha Excel):", data.loc[data['HORA'].str.startswith('1899'), 'HORA'].iloc[0])

In [ ]:
# FECHA: pandas reconoce ambos formatos automáticamente con format='mixed'
data['FECHA'] = pd.to_datetime(data['FECHA'], format='mixed', dayfirst=True)
data['AÑO'] = data['FECHA'].dt.year
data['MES'] = data['FECHA'].dt.month
print(data['FECHA'].min(), "→", data['FECHA'].max())

In [ ]:
# HORA: función que reconcilia los dos formatos y devuelve la hora del día (0-23)
def parse_hora(valor):
    valor = str(valor)
    if valor.startswith('1899-12-31'):          # formato pseudo-fecha de Excel
        return pd.to_datetime(valor, errors='coerce')
    else:                                        # formato texto en español "hh:mm:ss a./p. m."
        valor_en = valor.replace('a. m.', 'AM').replace('p. m.', 'PM')
        return pd.to_datetime(valor_en, format='%I:%M:%S %p', errors='coerce')

data['HORA_DIA'] = data['HORA'].apply(parse_hora).dt.hour
print("Registros sin poder interpretar la hora:", data['HORA_DIA'].isna().sum())
data['HORA_DIA'].describe()

**Resultado de la integración:** los 89.826 registros originales correspondían a 43.759 siniestros únicos. `FECHA` y `HORA` quedaron unificadas en columnas consistentes (`AÑO`, `MES`, `HORA_DIA`) sin ningún valor sin interpretar (0 fallos de parseo).

In [ ]:
# Ajustamos tipos de dato: categorías para las variables categóricas
# (CAUSA y BARRIO se dejan como texto por ahora: se agrupan en el paso 2 antes de tipificarlas)
for col in ['DÍA DE LA SEMANA', 'ESTADO DE BEODEZ', 'GRAVEDAD', 'CLASE DE ACCIDENTE',
            'AREA', 'AÑO', 'MES']:
    data[col] = data[col].astype('category')

data.info()

# 2. Eliminar variables irrelevantes y redundantes

**Criterio:**
- *Irrelevantes*: identificadores, texto libre, o columnas con datos insuficientes para aportar señal.
- *Redundantes*: columnas que quedaron duplicadas por una transformación matemática (p. ej. `FECHA`/`HORA` ya se descompusieron en `AÑO`, `MES`, `HORA_DIA`).

Primero revisamos el % de nulos de cada columna, porque varias columnas son candidatas a eliminarse por falta casi total de datos.

In [ ]:
nulos_pct = (data.isnull().sum() / len(data) * 100).round(2).sort_values(ascending=False)
nulos_pct

Cuatro columnas (`CLASE DE VEHICULO`, `TIPO DE SERVICIO`, `TIPO DE VICTIMA`, `SEXO`) tienen más del 99% de nulos: prácticamente no tienen información y no aportan como variables predictoras. `DIRECCIÓN` (98%) y `Coordenadas` (75%) también superan ampliamente el umbral de nulos manejable y, además, son identificadores de ubicación / texto libre, no variables analíticas directas. Las eliminamos por **irrelevantes**.

`RADICADO` es un identificador (ya cumplió su función para deduplicar) y `FECHA`/`HORA` quedaron **redundantes** frente a `AÑO`, `MES`, `HORA_DIA` que ya calculamos.

In [ ]:
# Variables irrelevantes (identificador, texto libre, o >75% de nulos)
irrelevantes = ['RADICADO', 'DIRECCIÓN', 'Coordenadas',
                 'CLASE DE VEHICULO', 'TIPO DE SERVICIO', 'TIPO DE VICTIMA', 'SEXO']
data.drop(columns=irrelevantes, inplace=True)

# Variables redundantes: ya extrajimos toda su información en AÑO, MES, HORA_DIA
redundantes = ['FECHA', 'HORA']
data.drop(columns=redundantes, inplace=True)

data.info()

## 2.1 Reducción de cardinalidad en `CAUSA` y `BARRIO`

`CAUSA` tiene 405 categorías distintas y `BARRIO` 49. Con esa cardinalidad, la mayoría de categorías tienen muy pocos registros (ruido estadístico) y, al convertirlas en variables *dummy* más adelante, generarían cientos de columnas casi vacías. Agrupamos cada variable en sus categorías más frecuentes + una categoría `OTRA(O)` para el resto — así conservamos el patrón dominante sin la dispersión.

In [ ]:
print("CAUSA - categorías únicas:", data['CAUSA'].nunique())
print("BARRIO - categorías únicas:", data['BARRIO'].nunique())

# Cuánto explican las 15 categorías más frecuentes de cada una
top_causas_pct = data['CAUSA'].value_counts(normalize=True).nlargest(15).sum() * 100
top_barrios_pct = data['BARRIO'].value_counts(normalize=True).nlargest(15).sum() * 100
print(f"\nLas 15 causas más frecuentes explican el {top_causas_pct:.1f}% de los registros")
print(f"Los 15 barrios más frecuentes explican el {top_barrios_pct:.1f}% de los registros")

In [ ]:
# CAUSA -> top 15 + 'OTRA' (conservamos NaN donde ya era NaN)
top_causas = data['CAUSA'].value_counts().nlargest(15).index
data['CAUSA_AGRUPADA'] = data['CAUSA'].where(data['CAUSA'].isin(top_causas), other='OTRA')

# BARRIO -> top 15 + 'OTRO'
top_barrios = data['BARRIO'].value_counts().nlargest(15).index
data['BARRIO_AGRUPADO'] = data['BARRIO'].where(data['BARRIO'].isin(top_barrios), other='OTRO')

data.drop(columns=['CAUSA', 'BARRIO'], inplace=True)  # quedan reemplazadas por las agrupadas
data['CAUSA_AGRUPADA'] = data['CAUSA_AGRUPADA'].astype('category')
data['BARRIO_AGRUPADO'] = data['BARRIO_AGRUPADO'].astype('category')
print("CAUSA_AGRUPADA:", data['CAUSA_AGRUPADA'].nunique(), "categorías")
print("BARRIO_AGRUPADO:", data['BARRIO_AGRUPADO'].nunique(), "categorías")

In [ ]:
data.head()

**Resultado:** partimos de 17 columnas originales. Tras eliminar 7 irrelevantes y 2 redundantes, y reemplazar `CAUSA`/`BARRIO` por sus versiones agrupadas, quedamos con **10 variables** (incluida la variable objetivo `GRAVEDAD`), todas con información útil y con cardinalidad manejable.

# 3. Descripción estadística de los datos

**Variables numéricas**

In [ ]:
data.describe()

In [ ]:
# Histogramas
data.hist(figsize=(10, 6))
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot conjunto de las numéricas
data.plot.box(figsize=(6, 5))
plt.show()

In [ ]:
data['RESULTADO DE BEODEZ'].plot.box()
plt.title('RESULTADO DE BEODEZ')
plt.show()

**Variables categóricas** (`value_counts()`)

In [ ]:
data['GRAVEDAD'].value_counts().plot(kind='bar', color=['#4C72B0','#DD8452','#C44E52'])
plt.title('GRAVEDAD (variable objetivo)')
plt.show()
data['GRAVEDAD'].value_counts(normalize=True).round(3) * 100

In [ ]:
data['DÍA DE LA SEMANA'].value_counts().plot(kind='bar')
plt.title('DÍA DE LA SEMANA')
plt.show()

In [ ]:
data['CLASE DE ACCIDENTE'].value_counts().plot(kind='bar')
plt.title('CLASE DE ACCIDENTE')
plt.show()

In [ ]:
data['CAUSA_AGRUPADA'].value_counts().plot(kind='barh')
plt.title('CAUSA (agrupada)')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
data['BARRIO_AGRUPADO'].value_counts().plot(kind='barh')
plt.title('BARRIO (agrupado)')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
data['AREA'].value_counts().plot(kind='bar')
plt.title('AREA')
plt.show()

In [ ]:
data['HORA_DIA'].value_counts().sort_index().plot(kind='bar')
plt.title('Distribución por hora del día')
plt.show()

In [ ]:
data['AÑO'].value_counts().sort_index().plot(kind='bar')
plt.title('AÑO')
plt.show()

In [ ]:
data['MES'].value_counts().sort_index().plot(kind='bar')
plt.title('MES')
plt.show()

In [ ]:
data['ESTADO DE BEODEZ'].value_counts().plot(kind='bar')
plt.title('ESTADO DE BEODEZ (0=No, 1=Sí)')
plt.show()

**Gráfica de relaciones entre variables numéricas**

In [ ]:
pd.plotting.scatter_matrix(data[['ESTADO DE BEODEZ', 'RESULTADO DE BEODEZ', 'HORA_DIA']], figsize=(8, 8))
plt.show()

**Perfilado de datos**

In [ ]:
!pip install ydata-profiling
from ydata_profiling import ProfileReport
profile_data = ProfileReport(data, minimal=True, title="Perfilado - Siniestros Viales Envigado")
profile_data

In [ ]:
# Guardamos el perfilado en HTML (entregable del taller)
profile_data.to_file(output_file="perfilado_siniestros_viales_envigado.html")

## Dimensiones de la calidad de datos

- **Completitud:** buena en las variables que conservamos; los nulos reales detectados son mínimos (<0.01%) una vez descartadas las columnas casi vacías (`CLASE DE VEHICULO`, `TIPO DE SERVICIO`, `TIPO DE VICTIMA`, `SEXO`, ya eliminadas en el paso 2).
- **Exactitud:** aceptable; no se detectan valores fuera de rango en las variables numéricas (`ESTADO DE BEODEZ` ∈ {0,1}, `RESULTADO DE BEODEZ` ∈ {0,1,2,3}).
- **Conformidad:** baja en el formato crudo de `FECHA`/`HORA` (dos formatos distintos conviviendo en la misma columna); ya resuelta en el paso 1.
- **Unicidad:** el archivo original no era único a nivel de accidente (46.067 filas duplicadas por `RADICADO`); resuelto en el paso 1.
- **Actualidad:** los datos cubren 2010–2022; no se puede verificar qué tan actualizado está el proceso de captura sin metadatos adicionales del portal.
- **Consistencia:** algunos valores tipo "NO REPORTADA"/"NO REPORTA" funcionan como nulos disfrazados; se tratan en el paso 4.


# 4. Limpieza de datos atípicos (errores)

Aquí solo corregimos **errores** de captura, no valores extremos legítimos (p. ej. un mes con más accidentes que otros no es un error, es un patrón real). Verificamos:
1. Que las variables numéricas estén dentro de su rango válido.
2. Que no existan valores tipo "no reportado" escondidos como si fueran una categoría más (son en realidad datos faltantes).

In [ ]:
# 1) Rango válido de las variables numéricas/codificadas
print("ESTADO DE BEODEZ - valores:", sorted(data['ESTADO DE BEODEZ'].unique()))
print("RESULTADO DE BEODEZ - valores:", sorted(data['RESULTADO DE BEODEZ'].unique()))
print("HORA_DIA - rango:", data['HORA_DIA'].min(), "-", data['HORA_DIA'].max())
print("AÑO - rango:", data['AÑO'].astype(int).min(), "-", data['AÑO'].astype(int).max())
# Todos dentro de rangos esperados: no hay errores de captura en estas columnas.

In [ ]:
# 2) Valores "no reportado" ocultos en variables categóricas
for col in ['CLASE DE ACCIDENTE', 'AREA', 'CAUSA_AGRUPADA', 'BARRIO_AGRUPADO']:
    encontrados = data[col].astype(str).str.upper().str.contains('NO REPORT', na=False)
    if encontrados.sum() > 0:
        print(f"{col}: {encontrados.sum()} registro(s) con valor 'no reportado'")

In [ ]:
# Los tratamos como error de captura -> los convertimos a nulo (NaN) para tratarlos en el paso 5
data['CLASE DE ACCIDENTE'] = data['CLASE DE ACCIDENTE'].astype(str).replace('NO REPORTADA', np.nan)
data['AREA'] = data['AREA'].astype(str).replace('NO REPORTADA', np.nan)
data['CAUSA_AGRUPADA'] = data['CAUSA_AGRUPADA'].astype(str).replace('NO REPORTA', np.nan)
data['BARRIO_AGRUPADO'] = data['BARRIO_AGRUPADO'].astype(str).replace('NO REPORTA', np.nan)

data.isnull().sum()

**Conclusión:** el dataset no tenía errores de rango en las variables numéricas. Los únicos errores encontrados fueron 5 registros (de 43.759, un 0,011%) con el valor placeholder "NO REPORTADA/NO REPORTA" en variables categóricas, que convertimos en nulos explícitos para imputarlos en el siguiente paso.

# 5. Limpieza de datos nulos: Imputación

**Estrategia:**
- Eliminar registros con más de 15% de nulos → no aplica (ningún registro tiene múltiples nulos).
- Eliminar columnas con más de 15-20% de nulos → ya se aplicó en el paso 2 (columnas con 99%+ de nulos).
- Imputar por moda las columnas categóricas restantes, dado que su % de nulos es mínimo.

In [ ]:
nulos_pct = (data.isnull().sum() / len(data) * 100).round(4)
nulos_pct[nulos_pct > 0]

In [ ]:
from sklearn.impute import SimpleImputer

cols_a_imputar = ['CLASE DE ACCIDENTE', 'AREA', 'CAUSA_AGRUPADA', 'BARRIO_AGRUPADO']
imputador = SimpleImputer(strategy='most_frequent')
data[cols_a_imputar] = imputador.fit_transform(data[cols_a_imputar])

print("Valor de la imputación (moda) por variable:")
for col, valor in zip(cols_a_imputar, imputador.statistics_):
    print(f"  {col}: '{valor}'")

In [ ]:
print("Nulos restantes en todo el dataset:", data.isnull().sum().sum())

# Recuperamos el tipo categórico (SimpleImputer devuelve texto)
for col in ['DÍA DE LA SEMANA', 'ESTADO DE BEODEZ', 'GRAVEDAD', 'CLASE DE ACCIDENTE',
            'CAUSA_AGRUPADA', 'BARRIO_AGRUPADO', 'AREA', 'AÑO', 'MES']:
    data[col] = data[col].astype('category')

data.info()

# 6. Correlaciones para redundancias

**Redundancia = correlación mayor a |0.8|.** Para calcular correlaciones todas las variables deben ser numéricas, así que generamos *dummies* de las categóricas (solo para este análisis; el dataset de trabajo `data` no se modifica todavía).

In [ ]:
cat_cols = ['DÍA DE LA SEMANA', 'CLASE DE ACCIDENTE', 'AÑO', 'MES',
            'CAUSA_AGRUPADA', 'BARRIO_AGRUPADO', 'GRAVEDAD']
data_num = pd.get_dummies(data, columns=cat_cols, drop_first=False, dtype=int)

# Área terminó como binaria porque se le imputaron la categoría de nulos (Ver 4 y 5)
# Estado de Beodez es binaria desde el principio
data_num = pd.get_dummies(data_num, columns=['AREA', 'ESTADO DE BEODEZ'], drop_first=True, dtype=int)
data_num.head()

In [ ]:
correlaciones = data_num.corr()
correlaciones

In [ ]:
plt.figure(figsize=(14, 12))
sns.heatmap(correlaciones, cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Matriz de correlaciones')
plt.show()

In [ ]:
# Pares con correlación absoluta > 0.8 (excluyendo la diagonal)
pares_altos = []
cols = correlaciones.columns
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        v = correlaciones.iloc[i, j]
        if abs(v) > 0.8:
            pares_altos.append((cols[i], cols[j], round(v, 3)))

pd.DataFrame(pares_altos, columns=['Variable 1', 'Variable 2', 'Correlación'])

Aparecen dos pares:

1. **`ESTADO DE BEODEZ_1` vs `RESULTADO DE BEODEZ` (0.899):** redundancia real. `RESULTADO DE BEODEZ` (0=negativo, 1-3=niveles de embriaguez) ya contiene la información de `ESTADO DE BEODEZ_1` (0/1) como caso particular — es prácticamente la misma variable con más detalle. Eliminamos `ESTADO DE BEODEZ` en `data` y conservamos `RESULTADO DE BEODEZ`, que es más informativa.
2. **`AREA_URBANA` vs `BARRIO_AGRUPADO_VEREDA PALMAS` (-0.871):** esta es una coincidencia esperada por definición geográfica — una "vereda" es por definición zona rural, así que ese barrio específico está perfectamente asociado a `AREA=RURAL`. No es una redundancia de la variable `AREA` completa (los otros 15 barrios no reproducen esa relación), así que **no** eliminamos `AREA` por este hallazgo puntual.

In [ ]:
# Eliminamos la variable redundante confirmada
data.drop(columns=['ESTADO DE BEODEZ'], inplace=True)
data.columns.tolist()

# 7. Correlaciones para irrelevancia con la variable objetivo

Buscamos variables cuya correlación con `GRAVEDAD` sea muy baja (0.0-0.05): si una variable casi no se mueve junto con el objetivo, aporta poco para predecirlo.

Como `GRAVEDAD` tiene 3 categorías, al hacer *dummy* (con `drop_first`) quedan 2 columnas objetivo (la categoría base, la primera en orden alfabético, no genera columna). Revisamos la correlación de cada predictora frente a ambas.

In [ ]:
correlaciones = data_num.corr()

cols_objetivo = [c for c in correlaciones.columns if c.startswith('GRAVEDAD_')]
print("Columnas objetivo (dummy):", cols_objetivo)
cor_variable_obj = correlaciones[cols_objetivo].drop(index=cols_objetivo)
cor_variable_obj.sort_values(by=cols_objetivo[0], key=abs, ascending=False).head(15)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 10))
for ax, col in zip(axes, cols_objetivo):
    s = cor_variable_obj[col].sort_values()
    ax.barh(s.index, s.values)
    ax.set_title(f'Correlación con {col}')
plt.tight_layout()
plt.show()

**Nota:** la correlación con `GRAVEDAD_MUERTOS` es baja en general para *todas* las variables — es esperable, porque MUERTOS representa apenas el 0.3% de los registros (muy poca variabilidad estadística para correlacionar linealmente). Una correlación baja aquí no siempre implica irrelevancia real, sino falta de casos.

Para decidir qué eliminar agregamos la correlación máxima absoluta de cada variable original (agrupando sus *dummies*) frente al objetivo.

In [ ]:
variables_originales = ['DÍA DE LA SEMANA', 'RESULTADO DE BEODEZ', 'CLASE DE ACCIDENTE',
                          'AREA', 'AÑO', 'MES', 'HORA_DIA', 'CAUSA_AGRUPADA', 'BARRIO_AGRUPADO']

def variable_base(col):
    for v in variables_originales:
        if col == v or col.startswith(v + '_'):
            return v
    return col

resumen = {}
for col in cor_variable_obj.index:
    base = variable_base(col)
    valor = cor_variable_obj.loc[col].abs().max()
    resumen.setdefault(base, []).append(valor)

resumen_df = pd.DataFrame({k: [max(v)] for k, v in resumen.items()}, index=['correlación_máx_abs']).T
resumen_df.sort_values('correlación_máx_abs')

`HORA_DIA` (0.011), `DÍA DE LA SEMANA` (0.017) y `MES` (0.021) son las variables con menor correlación con el objetivo. De estas, `AÑO` (0.043) es la que decidimos eliminar: además de tener una correlación baja, incluir el año exacto como predictor arriesga sobreajustar el modelo a los años observados (2010-2022) y perjudicar su capacidad de generalizar a años futuros — un argumento adicional al puramente estadístico.

`HORA_DIA`, `DÍA DE LA SEMANA` y `MES` se conservan: son variables cíclicas (la hora 23 está "cerca" de la hora 0, diciembre está "cerca" de enero) y la correlación lineal de Pearson subestima ese tipo de relaciones no monótonas, así que un valor bajo no es evidencia suficientemente fuerte para descartarlas.

In [ ]:
data.drop(columns=['AÑO'], inplace=True)
print("Variables predictoras finales:", [c for c in data.columns if c != 'GRAVEDAD'])
print("Total variables predictoras:", data.shape[1] - 1)

# 8. Balanceo de la variable objetivo (Clasificación)

`GRAVEDAD` está muy desbalanceada: cerca del 79.5 % de los siniestros son "SOLO DAÑOS", el 20.2 % "HERIDOS" y apenas el 0.3 % "MUERTOS". Un modelo entrenado así tendería a ignorar las clases minoritarias. Decisiones:

1. **Se elimina la clase MUERTOS:** con el 0.3 % de los registros no hay suficientes casos reales para aprender un patrón confiable, y sobremuestrearla implicaría inventar casi todos sus registros. El problema pasa a ser **binario: SOLO DAÑOS vs HERIDOS**.
2. **Submuestreo de SOLO DAÑOS al 75 %** (`RandomUnderSampler`): se descarta al azar la mitad de los registros de la clase mayoritaria.
3. **Sobremuestreo de HERIDOS** con **SMOTENC** (variante de SMOTE que soporta variables categóricas + numéricas mezcladas) hasta llevar a SOLO DAÑOS y HERIDOS hasta una proporción de 2:1.

In [ ]:
conteo = data['GRAVEDAD'].value_counts()
porcentaje = data['GRAVEDAD'].value_counts(normalize=True).round(3) * 100
ax = conteo.plot(kind='bar', color=['#4C72B0', '#DD8452', '#C44E52'])
plt.title('GRAVEDAD antes del balanceo')
plt.show()
porcentaje

In [ ]:
# Eliminamos la clase MUERTOS: el problema pasa a ser binario
data = data[data['GRAVEDAD'] != 'MUERTOS'].copy()
data['GRAVEDAD'] = data['GRAVEDAD'].astype(str).astype('category')
print("Shape sin MUERTOS:", data.shape)
data['GRAVEDAD'].value_counts()

In [ ]:
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTENC

X = data.drop(columns=['GRAVEDAD'])
Y = data['GRAVEDAD'].astype(str)

# Copia de los datos reales (binarios, sin balancear) para la verificación de la sección 11
X_real = X.copy()
Y_real = Y.copy()

# 1) Submuestreo: SOLO DAÑOS se reduce a 3/4 de su valor original
n_danos = (Y == 'SOLO DAÑOS').sum()
rus = RandomUnderSampler(sampling_strategy={'SOLO DAÑOS': int(n_danos * 0.75)}, random_state=42)
X_sub, Y_sub = rus.fit_resample(X, Y)
print("Después del submuestreo:", Y_sub.value_counts().to_dict())

# 2) Sobremuestreo: HERIDOS sube con SMOTENC hasta igualar a SOLO DAÑOS
columnas_categoricas = ['DÍA DE LA SEMANA', 'CLASE DE ACCIDENTE', 'AREA', 'MES',
                         'CAUSA_AGRUPADA', 'BARRIO_AGRUPADO']
indices_categoricos = [X.columns.get_loc(c) for c in columnas_categoricas]
sm = SMOTENC(categorical_features=indices_categoricos, sampling_strategy=0.5, random_state=42)
X_bal, Y_bal = sm.fit_resample(X_sub, Y_sub)

print("Shape antes:", X.shape, "-> Shape después:", X_bal.shape)

In [ ]:
# Reconstruimos el dataframe balanceado.
# Se mezclan las filas: SMOTENC agrega los registros sintéticos al final y la validación
# cruzada de GridSearchCV (StratifiedKFold sin shuffle) los concentraría en los últimos folds.
data = pd.DataFrame(X_bal, columns=X.columns).reset_index(drop=True)
data['GRAVEDAD'] = pd.Series(Y_bal).reset_index(drop=True)
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

data['GRAVEDAD'].value_counts().plot(kind='bar', color=['#4C72B0', '#DD8452'])
plt.title('GRAVEDAD después del balanceo (submuestreo + SMOTENC)')
plt.show()
data['GRAVEDAD'].value_counts()

**Resultado:** se eliminó MUERTOS, SOLO DAÑOS se redujo al 75% de su tamaño original y HERIDOS se completó con registros sintéticos de SMOTENC hasta llegar a una proporción 2:1; por cada registro de HERIDOS hay dos de SOLO DAÑOS. Nos rehusamos a llegar al balance perfecto (50/50) por evitar introducir demasiados datos sintéticos en HERIDOS o eliminar demasiados registros en SOLO DAÑOS. En cualquier caso, se conservan `X_real` / `Y_real` (datos reales, binarios y sin balancear) para la verificación final de la sección 11.

# 9. Transformaciones

- Si el método de ML requiere variables **categóricas/discretas** (árboles, Bayes) → discretización (9.1).
- Si el método de ML requiere variables **numéricas y normalizadas** (redes neuronales, regresión, SVM, KNN, K-means) → normalización + dummies + label encoder (9.2 a 9.4).

Guardamos dos versiones del dataset balanceado: `data_cat` (para métodos categóricos) y `data` (para métodos numéricos). En la sección 11 los seis modelos (incluidos los de árboles) se entrenan con la versión numérica `data`, para compararlos sobre las mismas variables.

In [ ]:
data_cat = data.copy()

**`9.1 Discretización`**: transformación de número a categoría.

Las dos variables numéricas del dataset son `HORA_DIA` (0-23) y `RESULTADO DE BEODEZ` (0-3, ordinal).

In [ ]:
# HORA_DIA -> franja horaria con nombres de categoría
data_cat['HORA_DIA'] = pd.cut(
    data_cat['HORA_DIA'],
    bins=[-1, 5, 11, 17, 23],
    labels=['Madrugada', 'Mañana', 'Tarde', 'Noche']
)
data_cat['HORA_DIA'].value_counts().plot(kind='bar')
plt.title('HORA_DIA discretizada')
plt.show()

In [ ]:
# RESULTADO DE BEODEZ -> categoría binaria
data_cat['RESULTADO DE BEODEZ'] = pd.cut(
    data_cat['RESULTADO DE BEODEZ'],
    bins=[-1, 0, 3],
    labels=['Negativo', 'Positivo']
)
data_cat['RESULTADO DE BEODEZ'].value_counts().plot(kind='bar')
plt.title('RESULTADO DE BEODEZ discretizado')
plt.show()

In [ ]:
data_cat.head()

**`9.2 Normalización de variables numéricas`**

In [ ]:
from sklearn.preprocessing import MinMaxScaler

min_max_scaler = MinMaxScaler()
data[['HORA_DIA', 'RESULTADO DE BEODEZ']] = min_max_scaler.fit_transform(data[['HORA_DIA', 'RESULTADO DE BEODEZ']])
data[['HORA_DIA', 'RESULTADO DE BEODEZ']].describe()

**`9.3 Dummies:`** transformación de categorías a números (solo para el dataset numérico/normalizado)

In [ ]:
columnas_categoricas = ['DÍA DE LA SEMANA', 'CLASE DE ACCIDENTE', 'MES',
                         'CAUSA_AGRUPADA', 'BARRIO_AGRUPADO']
data = pd.get_dummies(data, columns=columnas_categoricas, drop_first=False, dtype=int)

# El área termina siendo binaria por imputar las "no reportadas"
data = pd.get_dummies(data, columns=['AREA'], drop_first=True, dtype=int)
data.head()

**`9.4 Label Encoder:`** solo para la variable objetivo

In [ ]:
from sklearn.preprocessing import LabelEncoder

labelencoder = LabelEncoder()
data['GRAVEDAD'] = labelencoder.fit_transform(data['GRAVEDAD'])
print("Clases codificadas:", dict(zip(labelencoder.classes_, labelencoder.transform(labelencoder.classes_))))
data.head()

# 10. Guardar los datos preparados

In [ ]:
data_cat.to_excel("datos_categoricos_discretizados.xlsx", index=False)
print("Guardado: datos_categoricos_discretizados.xlsx", data_cat.shape)

In [ ]:
data.to_excel("datos_numericos_normalizados.xlsx", index=False)
print("Guardado: datos_numericos_normalizados.xlsx", data.shape)

# 11. Modelos de clasificación con hiperparametrización

Se entrenan seis modelos: árbol de decisión, Random Forest, XGBoost, KNN, SVM y red neuronal. En lugar de dividir en entrenamiento/prueba (70/30), cada modelo se ajusta con **`GridSearchCV`** usando **validación cruzada de 10 folds** y la medida **f1 macro**.

Se usa el dataset numérico `data` de la sección 9 (balanceado, normalizado, con dummies y label encoder). Para cada modelo se revisa el overfitting/underfitting comparando el f1 promedio en entrenamiento (Train CV) y en validación (Test CV).

In [ ]:
# Configuración hiperparametrización
from sklearn.model_selection import GridSearchCV
scoring = 'f1_macro'
cv = 10

# Configuración de datos
X = data.drop('GRAVEDAD', axis=1)  # Variables predictoras
Y = data['GRAVEDAD']               # Variable objetivo

# Medida de evaluación del mejor modelo
medidas = pd.DataFrame(index=['f1 de la CV'])
print("X:", X.shape, "| registros por clase:", dict(zip(labelencoder.classes_, np.bincount(Y))))

## 11.1. Árbol de decisión

In [ ]:
from sklearn.tree import DecisionTreeClassifier
modelTree = DecisionTreeClassifier(random_state=42)

# Definir los hiperparámetros
criterion = ['entropy', 'gini']      # Índice de información
min_samples_leaf = [2, 10, 50, 100]  # Cantidad de registros por hoja
max_depth = [None, 10, 20, 50]       # Niveles de profundidad

# Hiperparametrización
param_grid = dict(criterion=criterion, min_samples_leaf=min_samples_leaf, max_depth=max_depth)
grid = GridSearchCV(estimator=modelTree, param_grid=param_grid, scoring=scoring, cv=cv,
                    return_train_score=True, refit=True, n_jobs=-1)
grid.fit(X, Y)

# Mejor modelo
modelTree = grid.best_estimator_

# Mejores parámetros
print(grid.best_params_)

# Revisar overfitting y underfitting
print("Train CV:", grid.cv_results_["mean_train_score"][grid.best_index_])
print("Test CV:", grid.cv_results_["mean_test_score"][grid.best_index_])

# Comparación de medidas
medidas['Tree'] = grid.best_score_
medidas

In [ ]:
# Mejor árbol (se dibujan solo los 3 primeros niveles para que sea legible)
from sklearn.tree import plot_tree
plt.figure(figsize=(20, 10))
plot_tree(modelTree, feature_names=X.columns.values, class_names=labelencoder.classes_,
          max_depth=3, rounded=True, filled=True, fontsize=8)
plt.show()

## 11.2. Random Forest

Respecto a la configuración inicial propuesta (`n_estimators` de 1 a 4, `max_depth` de 1 a 7 y `class_weight='balanced_subsample'`): `n_estimators` sube a 50-200 (1 a 4 árboles no forman un bosque útil), `max_depth` usa 10/20/None (profundidades de 1 a 7 tienden a subajustar) y se quita `class_weight` porque los datos ya están balanceados.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
modelRF = RandomForestClassifier(random_state=42, n_jobs=-1)

# Definir los hiperparámetros
n_estimators = [50, 100, 200]       # Cantidad de árboles
criterion = ['gini', 'entropy']     # Índice de información
max_depth = [None, 10, 20]          # Niveles de profundidad
min_samples_leaf = [2, 10]          # Cantidad de registros por hoja
max_features = ['sqrt']             # Variables candidatas en cada división

# Hiperparametrización
param_grid = dict(n_estimators=n_estimators, criterion=criterion, max_depth=max_depth,
                  min_samples_leaf=min_samples_leaf, max_features=max_features)
grid = GridSearchCV(estimator=modelRF, param_grid=param_grid, scoring=scoring, cv=cv,
                    return_train_score=True, refit=True, n_jobs=-1)
grid.fit(X, Y)

# Mejor modelo
modelRF = grid.best_estimator_
print(grid.best_params_)

# Revisar overfitting y underfitting
print("Train CV:", grid.cv_results_["mean_train_score"][grid.best_index_])
print("Test CV:", grid.cv_results_["mean_test_score"][grid.best_index_])

# Comparación de medidas
medidas['RF'] = grid.best_score_
medidas

## 11.3. XGBoost

Ensamble de árboles construidos de forma secuencial: cada árbol corrige los errores de los anteriores. `learning_rate` controla cuánto aporta cada árbol y `subsample` la fracción de registros que ve cada uno.

In [ ]:
from xgboost import XGBClassifier
modelXGB = XGBClassifier(eval_metric='logloss', tree_method='hist', random_state=42, n_jobs=-1)

# Definir los hiperparámetros
n_estimators = [100, 200]           # Cantidad de árboles
max_depth = [3, 6, 9]               # Profundidad de cada árbol
learning_rate = [0.05, 0.1, 0.3]    # Tasa de aprendizaje
subsample = [0.8, 1.0]              # Fracción de registros por árbol

# Hiperparametrización
param_grid = dict(n_estimators=n_estimators, max_depth=max_depth,
                  learning_rate=learning_rate, subsample=subsample)
grid = GridSearchCV(estimator=modelXGB, param_grid=param_grid, scoring=scoring, cv=cv,
                    return_train_score=True, refit=True, n_jobs=-1)
grid.fit(X, Y)

# Mejor modelo
modelXGB = grid.best_estimator_
print(grid.best_params_)

# Revisar overfitting y underfitting
print("Train CV:", grid.cv_results_["mean_train_score"][grid.best_index_])
print("Test CV:", grid.cv_results_["mean_test_score"][grid.best_index_])

# Comparación de medidas
medidas['XGBoost'] = grid.best_score_
medidas

## 11.4. KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
modelKNN = KNeighborsClassifier()

# Definir los hiperparámetros
n_neighbors = [5, 15, 27]           # Cantidad de vecinos
weights = ['uniform', 'distance']   # Peso del voto de cada vecino

# Hiperparametrización
param_grid = dict(n_neighbors=n_neighbors, weights=weights)
grid = GridSearchCV(estimator=modelKNN, param_grid=param_grid, scoring=scoring, cv=cv,
                    return_train_score=True, refit=True, n_jobs=-1)
grid.fit(X, Y)

# Mejor modelo
modelKNN = grid.best_estimator_
print(grid.best_params_)

# Revisar overfitting y underfitting
print("Train CV:", grid.cv_results_["mean_train_score"][grid.best_index_])
print("Test CV:", grid.cv_results_["mean_test_score"][grid.best_index_])

# Comparación de medidas
medidas['KNN'] = grid.best_score_
medidas

## 11.5. SVM

Es el modelo más costoso (SVC escala casi cuadráticamente con el número de registros), por eso se prueba un grid reducido. Puede tardar entre 20 y 40 minutos en Colab.

In [ ]:
from sklearn.svm import SVC
modelSVM = SVC(cache_size=512)

# Definir los hiperparámetros
kernel = ['rbf']    # Función kernel
C = [1, 10]         # Penalización por error

# Hiperparametrización
param_grid = dict(kernel=kernel, C=C)
grid = GridSearchCV(estimator=modelSVM, param_grid=param_grid, scoring=scoring, cv=cv,
                    return_train_score=True, refit=True, n_jobs=-1)
grid.fit(X, Y)

# Mejor modelo
modelSVM = grid.best_estimator_
print(grid.best_params_)

# Revisar overfitting y underfitting
print("Train CV:", grid.cv_results_["mean_train_score"][grid.best_index_])
print("Test CV:", grid.cv_results_["mean_test_score"][grid.best_index_])

# Comparación de medidas
medidas['SVM'] = grid.best_score_
medidas

## 11.6. Red neuronal

In [ ]:
from sklearn.neural_network import MLPClassifier
modelNN = MLPClassifier(early_stopping=True, max_iter=300, random_state=42)

# Definir los hiperparámetros
hidden_layer_sizes = [(15,), (64, 32)]   # Neuronas por capa oculta
activation = ['relu', 'logistic']        # Función de activación

# Hiperparametrización
param_grid = dict(hidden_layer_sizes=hidden_layer_sizes, activation=activation)
grid = GridSearchCV(estimator=modelNN, param_grid=param_grid, scoring=scoring, cv=cv,
                    return_train_score=True, refit=True, n_jobs=-1)
grid.fit(X, Y)

# Mejor modelo
modelNN = grid.best_estimator_
print(grid.best_params_)

# Revisar overfitting y underfitting
print("Train CV:", grid.cv_results_["mean_train_score"][grid.best_index_])
print("Test CV:", grid.cv_results_["mean_test_score"][grid.best_index_])

# Comparación de medidas
medidas['NN'] = grid.best_score_
medidas

## 11.7. Comparación de modelos

In [ ]:
print(medidas.T.sort_values('f1 de la CV', ascending=False))
medidas.T.sort_values('f1 de la CV').plot(kind='barh', legend=False)
plt.xlabel('f1 macro (validación cruzada, 10 folds)')
plt.title('Comparación de modelos')
plt.show()

modelos = {'Tree': modelTree, 'RF': modelRF, 'XGBoost': modelXGB,
           'KNN': modelKNN, 'SVM': modelSVM, 'NN': modelNN}
mejor_nombre = medidas.loc['f1 de la CV'].idxmax()
mejor_modelo = modelos[mejor_nombre]
print("Mejor modelo:", mejor_nombre, "| f1 CV:", round(medidas.loc['f1 de la CV', mejor_nombre], 4))

## 11.8. Verificación del mejor modelo con datos reales

En las secciones anteriores el balanceo (submuestreo + SMOTENC) se hizo **antes** de la validación cruzada. Esto hace que el f1 obtenido pueda ser **optimista** por dos razones:
1. **Fuga de información:** los registros sintéticos de HERIDOS se crean a partir de vecinos que luego caen en otros folds.
2. **Folds de validación artificiales:** se valida sobre datos 50/50 que incluyen registros sintéticos, mientras que en la realidad la proporción es cercana a 80/20 (y el f1 macro baja de forma natural cuando la clase minoritaria es escasa).

Para obtener una estimación realista, se reevalúa el mejor modelo con los datos reales (`X_real`, `Y_real`) y un `Pipeline` de imblearn que balancea **solo dentro del fold de entrenamiento**: el fold de validación siempre contiene accidentes reales con su proporción original. La diferencia entre ambos f1 combina los dos efectos anteriores.

*Nota:* si el mejor modelo es SVM, esta celda vuelve a entrenarlo 10 veces y puede tardar tanto como su búsqueda de hiperparámetros.

In [ ]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score
from imblearn.pipeline import Pipeline

y_real = labelencoder.transform(Y_real)
clase_danos = labelencoder.transform(['SOLO DAÑOS'])[0]

def mitad_danos(y):
    # SOLO DAÑOS se reduce a la mitad, calculado sobre el fold de entrenamiento
    return {clase_danos: int((y == clase_danos).sum() // 2)}

categoricas = ['DÍA DE LA SEMANA', 'CLASE DE ACCIDENTE', 'AREA', 'MES',
               'CAUSA_AGRUPADA', 'BARRIO_AGRUPADO']
numericas = ['HORA_DIA', 'RESULTADO DE BEODEZ']
preparacion = ColumnTransformer([
    ('numericas', MinMaxScaler(), numericas),
    ('categoricas', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), categoricas)
])

pipeline_real = Pipeline([
    ('submuestreo', RandomUnderSampler(sampling_strategy=mitad_danos, random_state=42)),
    ('smotenc', SMOTENC(categorical_features=[X_real.columns.get_loc(c) for c in categoricas],
                        random_state=42)),
    ('preparacion', preparacion),
    ('modelo', clone(mejor_modelo)),
])

cv_real = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
f1_real = cross_val_score(pipeline_real, X_real, y_real, cv=cv_real, scoring='f1_macro', n_jobs=-1)

f1_global = medidas.loc['f1 de la CV', mejor_nombre]
print(f"Mejor modelo: {mejor_nombre}")
print(f"f1 CV con balanceo global:                 {f1_global:.4f}")
print(f"f1 CV con datos reales (balanceo por fold): {f1_real.mean():.4f} ± {f1_real.std():.4f}")
print(f"Diferencia:                                {f1_global - f1_real.mean():.4f}")

## 11.9. Conclusiones

*Valores de una ejecución local con las mismas semillas (`random_state=42`); en Colab pueden variar levemente.*

| Modelo | Train CV | Test CV (f1 macro) | Brecha |
|---|---|---|---|
| KNN | 0.967 | **0.814** | 0.153 |
| Random Forest | 0.895 | 0.805 | 0.089 |
| XGBoost | 0.916 | 0.798 | 0.118 |
| Árbol | 0.911 | 0.767 | 0.144 |
| SVM | 0.822 | 0.765 | 0.057 |
| Red neuronal | 0.779 | 0.750 | 0.029 |

- **Mejor modelo:** KNN (27 vecinos, peso por distancia), con f1 de 0.814, seguido muy de cerca por Random Forest (0.805) y XGBoost (0.798). Los tres modelos de mejor desempeño son los más flexibles.
- **Overfitting:** KNN (brecha 0.153) y el árbol de decisión (0.144) son los que más memorizan los datos de entrenamiento. En KNN es esperable porque, con `weights='distance'`, cada registro de entrenamiento se predice a sí mismo. El árbol sin límite de profundidad y con hojas de 2 registros también tiende a sobreajustar. Random Forest reduce esa brecha a 0.089 al promediar 200 árboles.
- **Underfitting:** la red neuronal y la SVM tienen los f1 de entrenamiento más bajos (0.78 y 0.82) y una brecha pequeña. Generalizan de forma estable, pero no alcanzan a capturar todo el patrón con las configuraciones probadas.
- **Verificación con datos reales:** al balancear solo dentro de cada fold y validar con accidentes reales (proporción ~80/20), el f1 de KNN baja de 0.814 a **0.671 ± 0.007** (diferencia de 0.143). Parte de esa caída es la fuga de información de SMOTENC: KNN es especialmente sensible a ella, porque un registro sintético casi siempre tiene como vecino más cercano al registro real del que se generó. La otra parte viene de validar con la proporción real de clases. El valor realista para reportar es **0.67**; las cifras de la tabla sirven para comparar modelos entre sí, no como desempeño esperado en producción.

**Nota sobre el balanceo:** Árbol, Random Forest, XGBoost y SVM podrían manejar el desbalance sin remuestrear los datos (`class_weight='balanced'` en sklearn o `scale_pos_weight` en XGBoost), entrenando directamente con los datos reales. KNN y la red neuronal (`MLPClassifier` no acepta pesos por clase) sí requieren el balanceo. Se usó el mismo balanceo para los seis modelos para compararlos en igualdad de condiciones.

# **FIN**